# Attention Is All You Need — Exploratory Analysis

**ArXivist-generated exploratory notebook**  
Paper: [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
Generated: 2026-05-31

---

This notebook visualises the **internal representations** of the Transformer,
reproducing key analyses from the paper (Figures 3–5) and exploring:

1. **Sinusoidal Positional Encoding** heatmap (Section 3.5)
2. **Attention weight heatmaps** — all 8 heads across encoder layers
3. **Attention head specialisation** — different heads attend to different patterns
4. **Cross-attention** — how decoder positions attend to encoder positions
5. **Interactive attention explorer** (if ipywidgets is available)

---

> **Note:** Cells 1–4 (PE visualisation) run without a checkpoint.  
> Cells 5+ require a trained checkpoint. Set `CHECKPOINT_PATH` below.

In [ ]:
# Cell 1 — Setup
import sys, torch
from pathlib import Path

# Add src/ to path
repo_root = Path(".").resolve().parent
src_path  = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ⚠️  Set this to your trained checkpoint path
CHECKPOINT_PATH = str(repo_root / "checkpoints" / "checkpoint_averaged.pt")
CONFIG_PATH     = str(repo_root / "configs" / "base.yaml")
CHECKPOINT_AVAILABLE = Path(CHECKPOINT_PATH).exists()

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Checkpoint available: {CHECKPOINT_AVAILABLE}")
if not CHECKPOINT_AVAILABLE:
    print("\n⚠️  No checkpoint found. Cells 5+ will use a randomly-initialised model.")
    print("   To train: python train.py --config configs/base.yaml")
    print("   Visualisations will still work but attention patterns won't be meaningful.")

## Visualisation 1 — Sinusoidal Positional Encoding

**Section 3.5**

$$PE_{(pos,2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad
PE_{(pos,2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

Each position gets a unique fingerprint in the embedding space. The sinusoidal
structure allows the model to learn relative position offsets via linear transformations.

In [ ]:
# Cell 2 — Positional Encoding heatmap
import torch
import numpy as np

try:
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    from transformer.models.embeddings import PositionalEncoding

    D_MODEL  = 512
    MAX_POS  = 50

    pe_module = PositionalEncoding(d_model=D_MODEL, max_len=200, dropout=0.0)
    pe_matrix = pe_module.pe[0, :MAX_POS, :].detach().numpy()  # [50, 512]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Full heatmap
    im = axes[0].imshow(pe_matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0].set_xlabel('Embedding dimension')
    axes[0].set_ylabel('Position')
    axes[0].set_title('Sinusoidal PE — all positions × dimensions')
    plt.colorbar(im, ax=axes[0])

    # First 64 dims, showing wavelength progression
    im2 = axes[1].imshow(pe_matrix[:, :64], aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    axes[1].set_xlabel('Embedding dimension (first 64)')
    axes[1].set_ylabel('Position')
    axes[1].set_title('PE zoom — wavelengths increase with dimension')
    plt.colorbar(im2, ax=axes[1])

    plt.tight_layout()
    plt.savefig('pe_heatmap.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Similarity between positions
    fig2, ax = plt.subplots(figsize=(7, 6))
    norms = np.linalg.norm(pe_matrix, axis=1, keepdims=True)
    sim   = (pe_matrix @ pe_matrix.T) / (norms @ norms.T + 1e-9)
    im3 = ax.imshow(sim, cmap='Blues', vmin=0, vmax=1)
    ax.set_xlabel('Position j')
    ax.set_ylabel('Position i')
    ax.set_title('Cosine similarity between PE vectors\n(Nearby positions are most similar)')
    plt.colorbar(im3, ax=ax)
    plt.tight_layout()
    plt.savefig('pe_similarity.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("✓ Positional encoding visualisation complete")

except ImportError as e:
    print(f"matplotlib required for visualisations: pip install matplotlib")
except Exception as e:
    import traceback; traceback.print_exc()

## Visualisation 2 — Attention Weight Heatmaps

**Appendix, Figures 3–5 of the paper**

The paper visualises attention patterns and shows that different heads learn to
perform different syntactic and semantic functions:
- Some heads track **long-distance dependencies**
- Some heads perform **anaphora resolution** (e.g., attending to the antecedent of "its")
- Some heads appear to track **positional structure**

Below we visualise all 8 encoder self-attention heads for a toy sequence.

In [ ]:
# Cell 3 — Load model (trained or random)
import torch
try:
    from transformer.models.transformer import Transformer
    from transformer.utils.config import TransformerConfig

    config = TransformerConfig.from_yaml(CONFIG_PATH)
    model  = Transformer(config)

    if CHECKPOINT_AVAILABLE:
        ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded checkpoint: {Path(CHECKPOINT_PATH).name}")
        model_note = "trained"
    else:
        print("Using randomly-initialised model (attention patterns not meaningful).")
        model_note = "random init"

    model = model.to(device)
    model.eval()
    print(f"Model ready ({model_note}).")

except Exception as e:
    import traceback; traceback.print_exc()

In [ ]:
# Cell 4 — Extract attention weights with hooks
import torch
from transformer.utils.masking import MaskFactory

try:
    # We'll capture attention weights from all encoder layers
    enc_attn_weights = {}  # layer_idx → [B, h, T, T]
    dec_self_attn_weights  = {}
    dec_cross_attn_weights = {}

    hooks = []

    def make_enc_hook(layer_idx):
        def hook(module, input, output):
            # output = (context, attn_weights)
            if isinstance(output, tuple) and len(output) == 2:
                enc_attn_weights[layer_idx] = output[1].detach().cpu()
        return hook

    def make_dec_self_hook(layer_idx):
        def hook(module, input, output):
            if isinstance(output, tuple) and len(output) == 2:
                dec_self_attn_weights[layer_idx] = output[1].detach().cpu()
        return hook

    def make_dec_cross_hook(layer_idx):
        def hook(module, input, output):
            if isinstance(output, tuple) and len(output) == 2:
                dec_cross_attn_weights[layer_idx] = output[1].detach().cpu()
        return hook

    for i, layer in enumerate(model.encoder.layers):
        hooks.append(layer.self_attn.register_forward_hook(make_enc_hook(i)))

    for i, layer in enumerate(model.decoder.layers):
        hooks.append(layer.self_attn.register_forward_hook(make_dec_self_hook(i)))
        hooks.append(layer.cross_attn.register_forward_hook(make_dec_cross_hook(i)))

    # Run a toy sentence through the model
    # Using token IDs as a proxy for a sentence (real use needs tokenizer)
    VOCAB = config.data.vocab_size
    PAD   = config.data.pad_idx

    # Toy: "The cat sat on the mat ." — 7 tokens
    src_ids = torch.tensor([[10, 20, 30, 40, 10, 50, 60]], device=device)  # [1, 7]
    tgt_ids = torch.tensor([[1,  70, 80, 90, 100, 110, 2]], device=device) # [1, 7] with BOS/EOS

    src_mask = MaskFactory.make_padding_mask(src_ids, PAD)
    tgt_in   = tgt_ids[:, :-1]
    tgt_mask = MaskFactory.make_tgt_mask(tgt_in, PAD)

    with torch.no_grad():
        _ = model(src_ids, tgt_in, src_mask=src_mask, tgt_mask=tgt_mask)

    # Remove hooks
    for h in hooks:
        h.remove()

    print(f"Captured encoder attention from {len(enc_attn_weights)} layers")
    print(f"Captured decoder self-attn from {len(dec_self_attn_weights)} layers")
    print(f"Captured decoder cross-attn from {len(dec_cross_attn_weights)} layers")
    if enc_attn_weights:
        w0 = enc_attn_weights[0]
        print(f"Encoder layer 0 attn shape: {w0.shape}  ← [B, h, T_src, T_src]")
    print("✓ Attention weights extracted")

except Exception as e:
    import traceback; traceback.print_exc()

In [ ]:
# Cell 5 — Visualise all 8 encoder attention heads (Layer 5, like paper Figure 3)
try:
    import matplotlib.pyplot as plt
    import numpy as np

    # Use layer 5 (index 4, 0-based) — the paper shows layer 5 in Figures 3–5
    # Fall back to last available layer
    layer_idx = min(4, max(enc_attn_weights.keys()))
    attn = enc_attn_weights[layer_idx][0]  # [h, T, T] — batch item 0
    n_heads = attn.shape[0]
    T = attn.shape[1]

    # Pseudo-tokens matching our toy input (real use: tokenizer.decode)
    tokens = ["The", "cat", "sat", "on", "the", "mat", "."]
    tokens = tokens[:T]  # trim to actual sequence length

    cols = 4
    rows = (n_heads + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    axes = axes.flatten()

    for h in range(n_heads):
        ax = axes[h]
        w  = attn[h].numpy()  # [T, T]
        im = ax.imshow(w, cmap='Blues', vmin=0, vmax=w.max())
        ax.set_xticks(range(T))
        ax.set_yticks(range(T))
        ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
        ax.set_yticklabels(tokens, fontsize=9)
        ax.set_title(f'Head {h+1}', fontsize=11)
        plt.colorbar(im, ax=ax, fraction=0.046)

    # Hide unused subplots
    for i in range(n_heads, len(axes)):
        axes[i].set_visible(False)

    title = (f'Encoder Self-Attention — Layer {layer_idx+1} — All {n_heads} Heads\n'
             f'(Reproducing paper Figures 3–5)')
    fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('attention_heads_all.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"✓ All {n_heads} attention heads visualised for encoder layer {layer_idx+1}")
    if not CHECKPOINT_AVAILABLE:
        print("  Note: using random weights — patterns are noise. Train first for meaningful heads.")

except ImportError:
    print("matplotlib required: pip install matplotlib")
except Exception as e:
    import traceback; traceback.print_exc()

In [ ]:
# Cell 6 — Cross-attention heatmap: which source tokens does each decoder position attend to?
try:
    import matplotlib.pyplot as plt

    layer_idx = min(4, max(dec_cross_attn_weights.keys()))
    cross_attn = dec_cross_attn_weights[layer_idx][0]  # [h, T_tgt, T_src]
    n_heads, T_tgt, T_src = cross_attn.shape

    src_tokens = ["The", "cat", "sat", "on", "the", "mat", "."][:T_src]
    tgt_tokens = ["BOS", "Die", "Katze", "saß", "auf", "der"][:T_tgt]  # mock translation

    cols = min(4, n_heads)
    rows = (n_heads + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 3.5))
    if n_heads == 1:
        axes = [[axes]]
    axes = [ax for row in (axes if rows > 1 else [axes]) for ax in (row if cols > 1 else [row])]

    for h in range(n_heads):
        ax = axes[h]
        w  = cross_attn[h].numpy()  # [T_tgt, T_src]
        im = ax.imshow(w, cmap='Oranges', aspect='auto')
        ax.set_xticks(range(T_src))
        ax.set_yticks(range(T_tgt))
        ax.set_xticklabels(src_tokens, rotation=45, ha='right', fontsize=9)
        ax.set_yticklabels(tgt_tokens, fontsize=9)
        ax.set_xlabel('Source tokens')
        ax.set_ylabel('Target tokens')
        ax.set_title(f'Cross-Attn Head {h+1}')
        plt.colorbar(im, ax=ax, fraction=0.046)

    for i in range(n_heads, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle(f'Decoder Cross-Attention — Layer {layer_idx+1}\n'
                 'Rows=target positions, Cols=source positions', fontsize=12)
    plt.tight_layout()
    plt.savefig('cross_attention.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"✓ Cross-attention visualised — Layer {layer_idx+1}, {n_heads} heads")

except Exception as e:
    import traceback; traceback.print_exc()

In [ ]:
# Cell 7 — Average attention entropy per head (head specialisation metric)
# Entropy: low = peaked/selective attention, high = diffuse/uniform attention
try:
    import matplotlib.pyplot as plt
    import numpy as np

    def head_entropy(attn_weights_dict, T):
        """Compute average attention entropy per head across all layers."""
        results = {}  # layer -> [h_entropy]
        for layer_idx, w in sorted(attn_weights_dict.items()):
            w_np = w[0].numpy()  # [h, T_q, T_k]
            # Entropy per position: -sum(p * log(p+eps))
            eps = 1e-9
            ent = -(w_np * np.log(w_np + eps)).sum(-1)  # [h, T_q]
            results[layer_idx] = ent.mean(-1)  # [h] — avg over positions
        return results

    enc_entropy = head_entropy(enc_attn_weights, T_src)

    if enc_entropy:
        n_layers = len(enc_entropy)
        n_heads  = len(list(enc_entropy.values())[0])

        entropy_matrix = np.stack([enc_entropy[i] for i in range(n_layers)])  # [L, h]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        im = ax1.imshow(entropy_matrix, cmap='YlOrRd', aspect='auto')
        ax1.set_xlabel('Attention head')
        ax1.set_ylabel('Encoder layer')
        ax1.set_title('Attention Entropy per Head\n(Low=selective, High=diffuse)')
        ax1.set_xticks(range(n_heads))
        ax1.set_xticklabels([f'H{i+1}' for i in range(n_heads)])
        ax1.set_yticks(range(n_layers))
        ax1.set_yticklabels([f'L{i+1}' for i in range(n_layers)])
        plt.colorbar(im, ax=ax1)

        # Average entropy per layer
        layer_avg = entropy_matrix.mean(-1)
        ax2.bar(range(n_layers), layer_avg, color='steelblue')
        ax2.set_xlabel('Layer')
        ax2.set_ylabel('Avg attention entropy (nats)')
        ax2.set_title('Average Entropy by Layer\n(Later layers often more selective)')
        ax2.set_xticks(range(n_layers))
        ax2.set_xticklabels([f'L{i+1}' for i in range(n_layers)])

        plt.tight_layout()
        plt.savefig('attention_entropy.png', dpi=120, bbox_inches='tight')
        plt.show()
        print("✓ Attention entropy analysis complete")
        print(f"Most selective head: Layer {entropy_matrix.min(axis=1).argmin()+1}, "
              f"Head {entropy_matrix.argmin() % n_heads + 1} "
              f"(entropy={entropy_matrix.min():.3f})")

except Exception as e:
    import traceback; traceback.print_exc()

In [ ]:
# Cell 8 — Encoder embedding PCA/TSNE visualisation (random model)
try:
    import matplotlib.pyplot as plt
    import numpy as np

    # Extract embedding matrix
    embed_weights = model.src_embedding.embedding.weight.detach().cpu().numpy()  # [V, d_model]
    V, D = embed_weights.shape
    print(f"Embedding matrix: {V:,} tokens × {D} dims")

    # PCA of first 500 embeddings
    sample = embed_weights[:500]
    mean   = sample.mean(0)
    centered = sample - mean
    U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    pc = U[:, :2] * S[:2]  # project onto first 2 PCs

    explained = (S[:2]**2).sum() / (S**2).sum()

    fig, ax = plt.subplots(figsize=(7, 6))
    scatter = ax.scatter(pc[:, 0], pc[:, 1], c=range(500),
                         cmap='viridis', alpha=0.6, s=15)
    plt.colorbar(scatter, ax=ax, label='Token ID')
    ax.set_xlabel(f'PC1 ({S[0]**2 / (S**2).sum() * 100:.1f}% variance)')
    ax.set_ylabel(f'PC2 ({S[1]**2 / (S**2).sum() * 100:.1f}% variance)')
    ax.set_title(f'PCA of Token Embeddings (first 500 tokens)\n'
                 f'Total explained: {explained*100:.1f}%'
                 + ('' if CHECKPOINT_AVAILABLE else ' [random init]'))
    plt.tight_layout()
    plt.savefig('embedding_pca.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"✓ Embedding PCA plotted (2 PCs explain {explained*100:.1f}% of variance)")

except Exception as e:
    import traceback; traceback.print_exc()

In [ ]:
# Cell 9 — Interactive attention explorer (requires ipywidgets)
try:
    import ipywidgets as widgets
    from IPython.display import display
    import matplotlib.pyplot as plt
    import numpy as np

    N_LAYERS = len(enc_attn_weights)
    N_HEADS  = enc_attn_weights[0].shape[1] if enc_attn_weights else 8
    T        = enc_attn_weights[0].shape[2] if enc_attn_weights else 7

    tokens = ["The", "cat", "sat", "on", "the", "mat", "."][:T]

    layer_slider = widgets.IntSlider(
        value=0, min=0, max=N_LAYERS-1, step=1,
        description='Layer:', continuous_update=False
    )
    head_slider = widgets.IntSlider(
        value=0, min=0, max=N_HEADS-1, step=1,
        description='Head:', continuous_update=False
    )
    attn_type = widgets.ToggleButtons(
        options=['Encoder Self-Attn', 'Decoder Cross-Attn'],
        description='Type:', button_style='info'
    )

    output = widgets.Output()

    def update_plot(change=None):
        with output:
            output.clear_output(wait=True)
            l = layer_slider.value
            h = head_slider.value
            atype = attn_type.value

            if atype == 'Encoder Self-Attn' and l in enc_attn_weights:
                w = enc_attn_weights[l][0, h].numpy()
                xlabel, ylabel = tokens, tokens
                cmap = 'Blues'
            elif atype == 'Decoder Cross-Attn' and l in dec_cross_attn_weights:
                w = dec_cross_attn_weights[l][0, h].numpy()
                src_tok = ["The", "cat", "sat", "on", "the", "mat", "."]
                tgt_tok = ["BOS", "Die", "Katze", "saß", "auf", "der"]
                xlabel, ylabel = src_tok[:w.shape[1]], tgt_tok[:w.shape[0]]
                cmap = 'Oranges'
            else:
                print(f"No data for Layer {l+1}, {atype}")
                return

            fig, ax = plt.subplots(figsize=(6, 5))
            im = ax.imshow(w, cmap=cmap, vmin=0)
            ax.set_xticks(range(len(xlabel)))
            ax.set_yticks(range(len(ylabel)))
            ax.set_xticklabels(xlabel, rotation=45, ha='right')
            ax.set_yticklabels(ylabel)
            ax.set_title(f'{atype} — Layer {l+1}, Head {h+1}')
            plt.colorbar(im, ax=ax)
            plt.tight_layout()
            plt.show()

    layer_slider.observe(update_plot, names='value')
    head_slider.observe(update_plot, names='value')
    attn_type.observe(update_plot, names='value')

    display(widgets.VBox([
        widgets.HTML('<h3>🔍 Interactive Attention Explorer</h3>'),
        attn_type, layer_slider, head_slider, output
    ]))
    update_plot()
    print("✓ Interactive widget ready — use sliders to explore layers and heads")

except ImportError:
    print("ipywidgets not available — run: pip install ipywidgets")
    print("Falling back to static plots (see cells above).")
except Exception as e:
    import traceback; traceback.print_exc()

## Summary of Visualisations

| Visualisation | File | Paper reference |
|---|---|---|
| Sinusoidal PE heatmap | `pe_heatmap.png` | Section 3.5 |
| PE cosine similarity | `pe_similarity.png` | Section 3.5 |
| All 8 encoder heads | `attention_heads_all.png` | Appendix Figures 3–5 |
| Decoder cross-attention | `cross_attention.png` | Section 3.2.3 |
| Attention entropy by layer | `attention_entropy.png` | Section 4 |
| Embedding PCA | `embedding_pca.png` | Section 3.4 |

---

For meaningful visualisations, **train the model first**:
```bash
python train.py --config configs/base.yaml
```
Then set `CHECKPOINT_PATH` in Cell 1 and re-run this notebook.